<a href="https://colab.research.google.com/github/Tmiller68/machine-learning-fundamentals/blob/week-7/Week_7_Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving creditcard.csv to creditcard.csv


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
df=pd.read_csv("creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
X=df.drop("Class", axis=1)
Y= df["Class"]
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [3]:
train_mean=X_train.mean()
train_std=X_train.std()

In [4]:
X_test_z=(X_test-train_mean)/train_std

In [5]:
X_test_z=X_test_z.abs()

In [6]:
anomaly_scores=X_test_z.max(axis=1)

In [7]:
pr_auc=average_precision_score(Y_test, anomaly_scores)
print(f"Z_score PR-AUC: {pr_auc}")


Z_score PR-AUC: 0.11414793180662038


The z-score anomaly detector achieved a PR-AUC of 0.114, which is much lower than the supervised Logistic Regression model's PR-AUC of about 0.74. Even though the score is much worse, the detector still identifies some fraud without using the fraud labels during training. It works because some fraudulent transactions contain feature values that are unusually far from the patterns seen in the training data. However, the z-score method is very simple and cannot learn more complex relationships between features the way a supervised model can.

I learned what a z-score is and how it can be used for anomaly detection. A z-score tells me how many standard deviations a value is away from the training-set mean, so a larger absolute z-score means that value is more unusual. For each transaction, I used the largest absolute z-score across its features as the anomaly score. This helped me understand how a model can flag unusual behavior without being trained directly on fraud labels.

What confused me at first was how the model could be called unsupervised if we still use y_test. The difference is that the labels are not used to create the anomaly scores or train the detector; they are only used after the fact to measure performance. I also had to understand why we use the maximum absolute z-score, which represents the most unusual feature in each transaction.

Day Two

In [8]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
import time

In [9]:
iso=IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

In [10]:
start_time=time.time()
iso.fit(X_train)
fit_time=time.time()-start_time

In [11]:
iso_scores = -iso.score_samples(X_test)

In [12]:
iso_pr_auc=average_precision_score(Y_test, iso_scores)
print(f"Isolation Forest PR-AUC: {iso_pr_auc}")
print(f"Fit time: {fit_time}")

Isolation Forest PR-AUC: 0.21797227754191925
Fit time: 0.8838818073272705


In [13]:
for n in [50, 100, 200]:
  iso=IsolationForest(
      n_estimators=n,
      contamination="auto",
      random_state=42
  )
  start_time=time.time()
  iso.fit(X_train)
  fit_time=time.time()-start_time
  iso_scores = -iso.score_samples(X_test)
  iso_pr_auc = average_precision_score(Y_test, iso_scores)

  print(f"n_estimators={n}")
  print(f"PR-AUC: {iso_pr_auc}")
  print(f"Fit time: {fit_time:.2f} seconds")
  print()

n_estimators=50
PR-AUC: 0.17517465326992007
Fit time: 0.27 seconds

n_estimators=100
PR-AUC: 0.21797227754191925
Fit time: 0.44 seconds

n_estimators=200
PR-AUC: 0.17171730448455547
Fit time: 0.87 seconds



In [14]:
for c in ["auto", 0.01, 0.05]:
    iso = IsolationForest(
        n_estimators=100,
        contamination=c,
        random_state=42
    )

    iso.fit(X_train)

    iso_scores = -iso.score_samples(X_test)
    iso_pr_auc = average_precision_score(Y_test, iso_scores)

    print(f"contamination={c}")
    print(f"PR-AUC: {iso_pr_auc}")
    print()

contamination=auto
PR-AUC: 0.21797227754191925

contamination=0.01
PR-AUC: 0.21797227754191925

contamination=0.05
PR-AUC: 0.21797227754191925



In [15]:
for c in ["auto", 0.01, 0.05]:
    iso = IsolationForest(
        n_estimators=100,
        contamination=c,
        random_state=42
    )

    iso.fit(X_train)

    iso_scores = -iso.score_samples(X_test)

    iso_pr_auc = average_precision_score(
        Y_test,
        iso_scores
    )

    print(f"contamination = {c}")
    print(f"PR-AUC = {iso_pr_auc:.4f}")
    print()

contamination = auto
PR-AUC = 0.2180

contamination = 0.01
PR-AUC = 0.2180

contamination = 0.05
PR-AUC = 0.2180



In [16]:
for n in [50, 100, 200]:
    iso = IsolationForest(
        n_estimators=n,
        contamination="auto",
        random_state=42
    )

    start_time = time.time()

    iso.fit(X_train)

    fit_time = time.time() - start_time

    iso_scores = -iso.score_samples(X_test)

    iso_pr_auc = average_precision_score(
        Y_test,
        iso_scores
    )

    print(f"n_estimators = {n}")
    print(f"PR-AUC = {iso_pr_auc:.4f}")
    print(f"Fit time = {fit_time:.2f} seconds")
    print()

n_estimators = 50
PR-AUC = 0.1752
Fit time = 0.24 seconds

n_estimators = 100
PR-AUC = 0.2180
Fit time = 0.43 seconds

n_estimators = 200
PR-AUC = 0.1717
Fit time = 1.19 seconds



At first, I was confused about what the contamination parameter actually changed. I learned that contamination mainly changes the cutoff for deciding which observations are labeled as anomalies, but it does not change the underlying anomaly scores used for PR-AUC. That is why changing contamination did not change my PR-AUC.

Changing the contamination value did not change the PR-AUC. I learned that contamination mainly changes the threshold used to decide which points are labeled as anomalies, while the continuous anomaly scores stay the same. Because PR-AUC is based on those scores rather than the hard predictions, the PR-AUC did not change.

Day Three

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score
import time
X_lof_train=X_train_scaled[:20000]
lof=LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
)
start_time=time.time()
lof.fit(X_lof_train)
lof_fit_time=time.time()-start_time
lof_scores=-lof.score_samples(X_test_scaled)
lof_pr_auc = average_precision_score(
    Y_test,
    lof_scores
)

print(f"LOF PR-AUC: {lof_pr_auc:.4f}")
print(f"LOF fit time: {lof_fit_time:.2f} seconds")

LOF PR-AUC: 0.0276
LOF fit time: 6.81 seconds


In [19]:
from sklearn.svm import OneClassSVM
X_ocsvm_train=X_train_scaled[:20000]

In [20]:
ocsvm = OneClassSVM(
    kernel="rbf",
    nu=0.01,
    gamma="scale"
)
start_time=time.time()
ocsvm.fit(X_ocsvm_train)
ocsvm_fit_time=time.time()-start_time
ocsvm_scores=-ocsvm.decision_function(X_test_scaled)
ocsvm_pr_auc = average_precision_score(
    Y_test,
    ocsvm_scores
)
print(f"OCSVM PR-AUC: {ocsvm_pr_auc:.4f}")
print(f"OCSVM fit time: {ocsvm_fit_time:.2f} seconds")

OCSVM PR-AUC: 0.0774
OCSVM fit time: 0.95 seconds


Isolation Forest performed the best so far with a PR-AUC of 0.2180, compared with 0.0276 for LOF and 0.0774 for One-Class SVM. LOF was also slower even when trained on a smaller sample, while One-Class SVM had to be sampled because it does not scale well to very large datasets. Based on both performance and scalability, Isolation Forest would be the most practical choice for a dataset with millions of rows.

I learned that LOF and One-Class SVM detect anomalies in different ways. LOF compares each point to the density of its nearby neighbors, while One-Class SVM tries to learn a boundary around normal-looking data. I also learned that both methods depend on distances, which is why scaling the features first is important. The results also showed me that a model can be accurate enough to test but still be impractical if it does not scale well to large datasets.

What confused me most was why LOF took so long to run even though the code itself was simple. I learned that LOF has to search for nearby neighbors for many observations, which becomes expensive as the dataset grows. I also had to understand why we sampled the training data for One-Class SVM instead of fitting it on the full dataset.

Day 4

In [21]:
X_train_benign = X_train[Y_train == 0]

In [22]:
print("Original training rows:", len(X_train))
print("Benign-only training rows:", len(X_train_benign))

Original training rows: 227845
Benign-only training rows: 227451


In [23]:
iso_benign = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)
start_time = time.time()
iso_benign.fit(X_train_benign)
iso_benign_fit_time = time.time() - start_time
iso_benign_scores=-iso_benign.score_samples(X_test)
iso_benign_pr_auc = average_precision_score(
    Y_test,
    iso_benign_scores
)
print(f"Benign-only Isolation Forest PR-AUC: {iso_benign_pr_auc:.4f}")
print(f"Fit time: {iso_benign_fit_time:.2f} seconds")

Benign-only Isolation Forest PR-AUC: 0.1741
Fit time: 0.73 seconds


In [25]:
benign_mean = X_train_benign.mean()
benign_std = X_train_benign.std()
X_test_z_benign = (X_test - benign_mean) / benign_std
X_test_z_benign = X_test_z_benign.abs()

z_benign_scores = X_test_z_benign.max(axis=1)
z_benign_pr_auc = average_precision_score(
    Y_test,
    z_benign_scores
)

print(f"Benign-only Z-score PR-AUC: {z_benign_pr_auc:.4f}")

Benign-only Z-score PR-AUC: 0.1357


Training only on benign data affected the detectors differently. The z-score baseline improved from a PR-AUC of 0.1140 to 0.1357, while Isolation Forest decreased from 0.2180 to 0.1741. Removing fraud likely helped the z-score method because its mean and standard deviation were based only on normal behavior, but Isolation Forest may have lost some useful structure from the full training distribution. Even though performance did not improve for every model, the benign-only protocol is a more realistic simulation of unseen attacks because the detector is trained without exposure to malicious examples.

I learned how novelty detection differs from just fitting an anomaly detector on all of the training data. In the train-on-benign setup, the model only learns from normal transactions and then has to identify unusual behavior in a mixed test set. I also learned that removing fraud from training does not automatically improve performance, because the z-score detector improved while Isolation Forest actually got worse.

What confused me at first was why we would remove fraud from the training data if we already had the labels available. I learned that the purpose is to simulate a real security situation where future attacks may be completely new and were never represented in training. I was also surprised that Isolation Forest performed worse after removing fraud, which showed me that the most realistic evaluation setup does not always give the highest score.